In [2]:
# ── Imports ────────────────────────────────────────────────────────────────────
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import statsmodels.api as sm
import numpy as np
import requests
from scipy.stats import pearsonr, spearmanr
from io import StringIO, BytesIO
import zipfile
from matplotlib.colors import ListedColormap


# ── Paramètres globaux ─────────────────────────────────────────────────────────
# dpi=120 : résolution des figures (120 points par pouce, plus net que le défaut)
plt.rcParams["figure.dpi"] = 120
# whitegrid : fond blanc avec grille légère ; palette "muted" : couleurs douces
sns.set_theme(style="whitegrid", palette="muted")
# Option pour pandas : formatage des flottants avec 2 décimales
pd.options.display.float_format = '{:.2f}'.format

# URL du fond de carte GeoJSON 
# contours des 96 départements métropolitains
GEOJSON_URL = "https://france-geojson.gregoiredavid.fr/repo/departements.geojson"

In [3]:
## !pip install great_tables

Import de la base de données avec les déplacements domicile - travail sur le site data gouv 
https://www.data.gouv.fr/datasets/deplacements-domicile-travail

In [4]:
# Utilisation de l'API pour recuperer les données 

url = "https://api.insee.fr/melodi/data/DS_RP_NAVETTES_PRINC"

r = requests.get(url)
data = r.json()

obs = data["observations"]

df = pd.json_normalize(obs)
df = df.rename(columns={
    "dimensions.GEO":"GEO",
    "dimensions.WORK_AREA":"WORK_AREA",
    "dimensions.FREQ":"FREQ",
    "dimensions.EMPSTA_ENQ":"EMPSTA_ENQ",
    "dimensions.TIME_PERIOD":"YEAR",
    "dimensions.RP_MEASURE":"MEASURE",
    "dimensions.TRANS":"TRANS",
    "dimensions.AGE":"AGE",
    "measures.OBS_VALUE_NIVEAU.value":"VALUE"
})

df_transport= df.copy()
print(df_transport.head(5))


                   GEO WORK_AREA FREQ EMPSTA_ENQ  YEAR MEASURE TRANS     AGE  \
0        2025-FRANCE-F        _T    A          1  2022     POP     2  Y_GE15   
1  2025-EPCI-249710047        10    A          1  2022     POP    _T  Y_GE15   
2  2025-EPCI-249710047        _T    A          1  2022     POP     6  Y_GE15   
3        2025-FRANCE-F        _T    A          1  2016     POP     1  Y_GE15   
4       2025-FRANCE-FM        23    A          1  2011     POP    _T  Y_GE15   

       VALUE  
0 1729514.24  
1    2045.00  
2     131.00  
3 1153683.83  
4  995946.60  


Avec une limitation de 10000 lignes par page de l'API nous ne pouvons pas recuperer toutes les données par consequent nous utilisons l'approche avec le fichier ZIP

In [5]:

url = "https://www.data.gouv.fr/api/1/datasets/r/b35881a9-da09-49bf-a80e-8fa17651e927"

try:
    # 1. On télécharge le ZIP
    response = requests.get(url)
    
    # 2. On ouvre l'archive en mémoire
    with zipfile.ZipFile(BytesIO(response.content)) as z:
        # 3. On ouvre spécifiquement le fichier de données
        with z.open('DS_RP_NAVETTES_PRINC_2022_data.csv') as f:
            df_transport = pd.read_csv(f, sep=';', encoding='latin-1', low_memory=False)
            
    print(f"Importation réussie ! {len(df_transport)} lignes chargées.")
    print(df_transport.head())

except Exception as e:
    print(f"Erreur : {e}")

Importation réussie ! 1191432 lignes chargées.
      AGE  EMPSTA_ENQ FREQ        GEO GEO_OBJECT RP_MEASURE TRANS WORK_AREA  \
0  Y_GE15           1    A          F     FRANCE        POP     2        _T   
1  Y_GE15           1    A  249710047       EPCI        POP    _T        10   
2  Y_GE15           1    A  249710047       EPCI        POP     6        _T   
3  Y_GE15           1    A          F     FRANCE        POP     1        _T   
4  Y_GE15           1    A         FM     FRANCE        POP    _T        23   

   TIME_PERIOD  OBS_VALUE  
0         2022 1729514.24  
1         2022    2045.00  
2         2022     131.00  
3         2016 1153683.83  
4         2011  995946.60  


Vérifications et prise en main de la base 

In [ ]:
print(df_transport.info())
print(df_transport.nunique())

In [6]:
# 1. Suppression des colonnes inutiles
df_transport = df_transport.drop(columns=['AGE', 'EMPSTA_ENQ', 'FREQ', 'RP_MEASURE'])

# 2. Renommer TIME_PERIOD en ANNEE
df_transport = df_transport.rename(columns={'TIME_PERIOD': 'ANNEE'})

# 3. Filtrer pour ne garder que les modalités spécifiques de GEO_OBJECT
df_transport = df_transport[df_transport['GEO_OBJECT'].isin(['COM', 'DEP', 'FRANCE', 'REG'])]

# 4. Vérification
print("Modalités restantes dans GEO_OBJECT :", df_transport['GEO_OBJECT'].unique())
print("Colonnes actuelles :", df_transport.columns.tolist())
print(f"Nombre de lignes restantes : {len(df_transport)}")

Modalités restantes dans GEO_OBJECT : ['FRANCE' 'COM' 'DEP' 'REG']
Colonnes actuelles : ['GEO', 'GEO_OBJECT', 'TRANS', 'WORK_AREA', 'ANNEE', 'OBS_VALUE']
Nombre de lignes restantes : 975848


In [ ]:
print("--- Fréquences pour TRANS ---")
print(df_transport['TRANS'].value_counts(dropna=False))

print("\n--- Fréquences pour WORK_AREA ---")
print(df_transport['WORK_AREA'].value_counts(dropna=False))

print("\n--- Fréquences pour GEO_OBJECT ---")
print(df_transport['GEO_OBJECT'].value_counts(dropna=False))

In [ ]:
# Liste des colonnes à analyser
colonnes_cibles = ['WORK_AREA', 'ANNEE', 'TRANS']

for col in colonnes_cibles:
    print(f"--- Modalités pour {col} ---")
    # Affiche les valeurs uniques
    print(df_transport[col].unique())
    # Affiche le nombre d'occurrences pour chaque modalité
    # print(df[col].value_counts())
    print("\n")

analyse des modes de transport 

In [7]:
print(df_transport.columns)
print("Modalités de TRANS :", df_transport['TRANS'].unique())
print("Modalités de ANNEE :", df_transport['ANNEE'].unique())
print("Modalités croisées :", pd.crosstab(df_transport['TRANS'], df_transport['ANNEE']))

Index(['GEO', 'GEO_OBJECT', 'TRANS', 'WORK_AREA', 'ANNEE', 'OBS_VALUE'], dtype='object')
Modalités de TRANS : ['2' '1' '_T' '4' '3' '5' '6' '3T4']
Modalités de ANNEE : [2022 2016 2011]
Modalités croisées : ANNEE    2011    2016    2022
TRANS                        
1           0   34088   33978
2           0   31408   31078
3           0       0   22122
3T4         0   27779       0
4           0       0   24531
5           0   34961   34961
6           0   27634   27606
_T     213872  215450  216380


In [8]:
# 1. Préparation de la base 
df_clean = df_transport.copy()
df_clean = df_clean.rename(columns={'OBS_VALUE': 'VALUE'})
df_clean['VALUE'] = pd.to_numeric(df_clean['VALUE'], errors='coerce')

print(df_clean.head(5))

         GEO GEO_OBJECT TRANS WORK_AREA  ANNEE      VALUE
0          F     FRANCE     2        _T   2022 1729514.24
3          F     FRANCE     1        _T   2016 1153683.83
4         FM     FRANCE    _T        23   2011  995946.60
5          F     FRANCE    _T        23   2011  997887.98
80004  06066        COM    _T        10   2022     131.96


In [9]:
# modes de transport par période 

# 1. ANNEE vs WORK_AREA
tab_work_area = pd.crosstab(
    df_transport['ANNEE'], 
    df_transport['WORK_AREA'], 
    dropna=False
)

# 2. ANNEE vs TRANS
tab_trans = pd.crosstab(
    df_transport['ANNEE'], 
    df_transport['TRANS'], 
    dropna=False
)

print("--- Tableau Croisé : Année vs Zone de Travail ---")
print(tab_work_area)

print("\n--- Tableau Croisé : Année vs Mode de Transport ---")
print(tab_trans)

--- Tableau Croisé : Année vs Zone de Travail ---
WORK_AREA     10  20_30     21     22     23  24T30      _T
ANNEE                                                      
2011       34882  34957  34940  28755  31311  14061   34966
2016       34882  34958  34939  31182  29918  14607  190834
2022       34856  34955  34939  31493  30358  14816  209239

--- Tableau Croisé : Année vs Mode de Transport ---
TRANS      1      2      3    3T4      4      5      6      _T
ANNEE                                                         
2011       0      0      0      0      0      0      0  213872
2016   34088  31408      0  27779      0  34961  27634  215450
2022   33978  31078  22122      0  24531  34961  27606  216380


On va utiliser que 2016 et 2022, pour 2011 nous n'avons pas le détail des modes de transport.
De plus, en 2016 les modalités 3 et 4 étaient regroupées

In [10]:
# 1. Filtrer pour ne garder que 2016 et 2022
df_final = df_transport[df_transport['ANNEE'].isin([2016, 2022])].copy()

# 2. Gérer le regroupement 3T4 pour l'année 2022
# On identifie les lignes où TRANS est 3 ou 4, et on les transforme en 3T4
df_final.loc[df_final['TRANS'].isin(['3', '4']), 'TRANS'] = '3T4'

# Après avoir renommé 3 et 4 en 3T4, on agrège les valeurs (OBS_VALUE) 
# pour éviter d'avoir deux lignes "3T4" par zone
df_final = df_final.groupby(['ANNEE', 'GEO', 'GEO_OBJECT', 'TRANS', 'WORK_AREA'], as_index=False)['OBS_VALUE'].sum()

# 3. Créer la variable libellé pour les modes de transport (Dictionnaire de correspondance)
mapping_trans = {
    '1': 'Pas de transport',
    '2': 'Marche à pied',
    '3': 'Vélo',
    '4': 'Deux-roues motorisés',
    '5': 'Voiture, camion ou fourgonnette',
    '6': 'Transport en commun',
    '3T4': 'Deux-roues', # Note : dans certains fichiers 3T4 = Voiture, vérifiez bien vos totaux
    '_T': 'Total'
}

df_final['LIB_TRANS'] = df_final['TRANS'].map(mapping_trans)

# 4. Vérification avec le nouveau tableau croisé
print("--- Vérification du regroupement par ANNEE ---")
print(pd.crosstab(df_final['ANNEE'], df_final['TRANS']))

--- Vérification du regroupement par ANNEE ---
TRANS      1      2    3T4      5      6      _T
ANNEE                                           
2016   34088  31408  27779  34961  27634  215450
2022   33978  31078  28228  34961  27606  216380


In [ ]:
# 4. Vérification 
print("--- Vérification du regroupement par ANNEE ---")
print(pd.crosstab(df_final['ANNEE'], df_final['LIB_TRANS']))

Faire deux tables, une pour 2016 et une pour 2022 avec des données agrégées pour OBS_VALUE par (croisement) département (filtrer GEO_OBJECT = DEP, puis les codes dpt se trouvent dans GEO) et modes de transport.
Puis calculer pour chaque département la part de chaque mode de transport dans le total.

In [11]:
# Liste des codes dep à supprimer (DOM)
dom_codes = ['971', '972', '973', '974', '976'] 

# Filtrage : on ne garde que les lignes où GEO n'est PAS dans la liste
df_final = df_final[~df_final['GEO'].isin(dom_codes)]

# 1. Filtrage pour ne garder que les Départements (DEP) et les années cibles
df_dep = df_final[(df_final['GEO_OBJECT'] == 'DEP') & 
                  (df_final['ANNEE'].isin([2016, 2022])) &
                  (df_final['TRANS'] != '_T')].copy()

# 2. Création des deux tables (2016 et 2022)
# On agrège OBS_VALUE par Département (GEO) et Mode de transport
tables_par_annee = {}

for annee in [2016, 2022]:
    # Pivot pour avoir les départements en lignes et les modes en colonnes
    pivot = df_dep[df_dep['ANNEE'] == annee].pivot_table(
        index=['GEO'], 
        columns=['TRANS', 'LIB_TRANS'], 
        values='OBS_VALUE', 
        aggfunc='sum',
        fill_value=0
    )
    
    # 3. Calcul de la part de chaque mode de transport (%)
    # On divise chaque cellule par la somme de sa ligne (le total du département)
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
    
    tables_par_annee[annee] = pivot_pct.round(2)

# --- AFFICHAGE DES RÉSULTATS ---

print("--- PARTS MODALES PAR DÉPARTEMENT EN 2016 (%) ---")
print(tables_par_annee[2016].head())

print("\n" + "="*50 + "\n")

print("--- PARTS MODALES PAR DÉPARTEMENT EN 2022 (%) ---")
print(tables_par_annee[2022].head())

--- PARTS MODALES PAR DÉPARTEMENT EN 2016 (%) ---
TRANS                    1             2        3T4  \
LIB_TRANS Pas de transport Marche à pied Deux-roues   
GEO                                                   
01                    4.21          4.91       2.65   
02                    5.24          7.00       2.18   
03                    6.93          6.81       2.87   
04                    5.92          8.83       1.99   
05                    6.39         12.11       2.48   

TRANS                                   5                   6  
LIB_TRANS Voiture, camion ou fourgonnette Transport en commun  
GEO                                                            
01                                  81.96                6.26  
02                                  80.29                5.29  
03                                  80.35                3.04  
04                                  80.39                2.86  
05                                  75.22                3.80

In [ ]:
# On s'assure de travailler sur les données filtrées pour les départements
df_dep_only = df_final[df_final['GEO_OBJECT'] == 'DEP']

for annee in sorted(df_final['ANNEE'].unique()):
    # Extraction des codes GEO uniques pour l'année en cours
    codes_geo = sorted(df_dep_only[df_dep_only['ANNEE'] == annee]['GEO'].unique())
    
    print(f"\n--- MODALITÉS GEO (DÉPARTEMENTS) EN {annee} ---")
    print(f"Nombre total : {len(codes_geo)}")
    print(codes_geo)

In [1]:
from fonctions import afficher_gt_part_modale
# AFFICHAGE

# Affichage pour 2016
gt_2016 = afficher_gt_part_modale(tables_par_annee[2016], 2016)
display(gt_2016)

# Affichage pour 2022
gt_2022 = afficher_gt_part_modale(tables_par_annee[2022], 2022)
display(gt_2022)

NameError: name 'tables_par_annee' is not defined

Calcul des évolutions 

In [ ]:

# 1. Calcul de la différence (2022 - 2016)
df_evol = tables_par_annee[2022] - tables_par_annee[2016]

print("--- ÉVOLUTION 2016-2022 (en points) ---")
print(df_evol.head())


Les départements avec les évolutions positives les plus importantes

In [ ]:
from fonctions import extraire_extremes

# 1. Extraction et nettoyage
# On récupère les colonnes d'évolution (en points de %)
evol_2roues = df_evol.xs('Deux-roues', axis=1, level=1)
evol_tc = df_evol.xs('Transport en commun', axis=1, level=1)

# --- AFFICHAGE ---
extraire_extremes(evol_2roues, "Deux-roues")
extraire_extremes(evol_tc, "Transports en commun")

In [ ]:

# Chargement du fond de carte 
# Le GeoJSON contient les contours géographiques des 96 départements métropolitains.
carte_base = gpd.read_file(GEOJSON_URL)

# 1. Préparation des données (on refait le merge pour être sûr)
# On s'assure que les codes GEO sont bien synchronisés
carte_base['code'] = carte_base['code'].astype(str).str.zfill(2)

# On crée les GeoDataFrames (C'est ici que map_2roues est défini)
map_2roues = carte_base.merge(evol_2roues, left_on='code', right_index=True)
map_tc = carte_base.merge(evol_tc, left_on='code', right_index=True)

# 2. Définition des seuils et couleurs
bins_2roues = [0, 0.5, 1, 2, 3] 
bins_tc = [-0.5, 0, 0.5, 1, 1.5]

# Verts pour les deux-roues
colors_2roues = ['#D9EF8B', '#A6D96A', '#66BD63', '#1A9850', '#006837']
cmap_2roues = ListedColormap(colors_2roues)

# Orange + Verts pour les TC
colors_tc = ['#FDAE61', '#D9EF8B', '#A6D96A', '#66BD63', '#1A9850']
cmap_tc = ListedColormap(colors_tc)

# 3. Affichage des cartes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

# Carte Deux-roues
map_2roues.plot(
    column=map_2roues.columns[-1], 
    cmap=cmap_2roues, 
    scheme='UserDefined',
    classification_kwds={'bins': bins_2roues},
    legend=True, 
    ax=ax1,
    legend_kwds={'title': "Points de %", 'loc': 'lower left', 'fmt': "{:.1f}"},
    edgecolor='black', linewidth=0.2
)
ax1.set_title("Évolution Deux-roues (2016-2022)", fontsize=14)
ax1.axis('off')

# Carte Transports en commun
map_tc.plot(
    column=map_tc.columns[-1], 
    cmap=cmap_tc, 
    scheme='UserDefined',
    classification_kwds={'bins': bins_tc},
    legend=True, 
    ax=ax2,
    legend_kwds={'title': "Points de %", 'loc': 'lower left', 'fmt': "{:.1f}"},
    edgecolor='black', linewidth=0.2
)
ax2.set_title("Évolution Transports en Commun (2016-2022)", fontsize=14)
ax2.axis('off')

plt.tight_layout()
plt.show()